In [1]:
import os
import math
import random
from typing import List, Dict
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoConfig,
)
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt", quiet=True)

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>


False

In [3]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

@dataclass
class Config:
    # Models (you can replace names if not available)
    code_model_name: str = "microsoft/CodeGPT-small-py"      # suggested code-specialized (GPT-like)
    general_model_name: str = "EleutherAI/gpt-neo-125M"      # suggested general-purpose
    code_classifier_name: str = "microsoft/codebert-base"    # code encoder for classification
    general_classifier_name: str = "bert-base-uncased"       # general encoder baseline

    # Data / training
    max_seq_len: int = 512
    gen_max_new_tokens: int = 128
    gen_train_samples: int = 2000     # reduce for quick runs; increase for serious runs
    gen_val_samples: int = 200
    cls_train_samples: int = 2000
    cls_val_samples: int = 400
    batch_size: int = 4                # keep small if GPU memory limited
    cls_batch_size: int = 16
    num_epochs_gen: int = 2
    num_epochs_cls: int = 3
    lr: float = 5e-5
    output_dir: str = "./results_llm_lab"

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)

Device: cpu


In [4]:
def normalize_code_text(s: str) -> str:
    """Minimal normalization for code comparison: strip trailing spaces, collapse multi-blank lines."""
    # keep indentation as it's meaningful for Python; just strip trailing whitespace lines
    lines = [ln.rstrip() for ln in s.splitlines()]
    # remove leading/trailing blank lines
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop(-1)
    return "\n".join(lines)

In [5]:
def prepare_code_generation_dataset(max_train=2000, max_val=200):
    """
    Attempts to load CodeSearchNet (python) via datasets library.
    Falls back to a tiny synthetic dataset if unavailable.
    Returns: (train_samples, val_samples) where each sample = dict(prompt, target)
    """
    print("Preparing code generation dataset...")
    samples_train = []
    samples_val = []
    try:
        ds = load_dataset("code_search_net", "python")
        # ds has splits 'train', 'validation', 'test' and inside 'function' items with 'docstring' and 'code'
        def extract_examples(split, max_n):
            res = []
            for i, item in enumerate(ds[split]):
                if i >= max_n:
                    break
                func = item.get("function") or item  # sometimes dataset nested
                doc = func.get("docstring", "")
                code = func.get("code", "")
                # skip empty
                if not doc or not code:
                    continue
                prompt = "### Docstring:\n" + doc.strip() + "\n### Code:\n"
                res.append({"prompt": prompt, "target": code})
            return res

        samples_train = extract_examples("train", max_train)
        samples_val = extract_examples("validation", max_val)
        print(f"Loaded CodeSearchNet python: train={len(samples_train)}, val={len(samples_val)}")
    except Exception as e:
        print("Could not load CodeSearchNet via datasets. Falling back to synthetic small dataset. Error:", e)
        # Tiny synthetic examples for demonstration; replace with real dataset for lab
        toy = [
            ("Add two numbers", "def add(a, b):\n    return a + b\n"),
            ("Return factorial", "def fact(n):\n    if n <= 1:\n        return 1\n    return n * fact(n-1)\n"),
            ("Check prime", "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5)+1):\n        if n % i == 0:\n            return False\n    return True\n"),
        ]
        for i in range(max_train):
            doc, code = random.choice(toy)
            prompt = "### Docstring:\n" + doc + "\n### Code:\n"
            if i < max_val:
                samples_val.append({"prompt": prompt, "target": code})
            else:
                samples_train.append({"prompt": prompt, "target": code})
    return samples_train, samples_val

In [6]:
class CodeGenDataset(Dataset):
    def __init__(self, examples: List[Dict], tokenizer, max_length=512):
        """
        examples: list of {"prompt":..., "target": ...}
        tokenizer: HF tokenizer for causal LM
        """
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return ex

def collate_codegen(batch, tokenizer, max_length):
    """
    Each batch item has 'prompt' and 'target'.
    We create input_ids = token(prompt + target)
    and labels = [-100]*len(prompt_ids) + target_ids (so prompt tokens are not included in loss)
    """
    input_ids_list = []
    labels_list = []
    attention_mask_list = []
    for ex in batch:
        prompt = ex["prompt"]
        target = ex["target"]
        # tokenize separately
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
        target_ids = tokenizer.encode(target, add_special_tokens=False)
        # truncate if too long
        total = prompt_ids + target_ids
        if len(total) > max_length:
            # prefer keeping prompt, truncate target
            allowed_target_len = max_length - len(prompt_ids)
            if allowed_target_len <= 0:
                # prompt itself too long: truncate prompt from left (rare)
                prompt_ids = prompt_ids[-(max_length//2):]
                allowed_target_len = max_length - len(prompt_ids)
            target_ids = target_ids[:allowed_target_len]
            total = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids[:]  # the model will compute loss only on target tokens
        input_ids_list.append(total)
        labels_list.append(labels)
        attention_mask_list.append([1] * len(total))

    # pad to same length
    batch_max_len = max(len(ids) for ids in input_ids_list)
    batch_max_len = min(batch_max_len, max_length)
    padded_input_ids = []
    padded_labels = []
    padded_attention = []
    for ids, labs, am in zip(input_ids_list, labels_list, attention_mask_list):
        pad_len = batch_max_len - len(ids)
        padded_input_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
        padded_labels.append(labs + [-100] * pad_len)
        padded_attention.append(am + [0] * pad_len)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
        "attention_mask": torch.tensor(padded_attention, dtype=torch.long),
    }

In [7]:
def train_causal_lm(model_name, train_examples, val_examples, config: Config, out_prefix="code"):
    print(f"\n--- Fine-tuning causal LM '{model_name}' ({out_prefix}) ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Ensure pad token exists (GPT2 family often doesn't)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})
    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    # resize token embeddings if tokenizer changed
    model.resize_token_embeddings(len(tokenizer))

    train_ds = CodeGenDataset(train_examples, tokenizer, max_length=config.max_seq_len)
    val_ds = CodeGenDataset(val_examples, tokenizer, max_length=config.max_seq_len)

    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              collate_fn=lambda b: collate_codegen(b, tokenizer, config.max_seq_len))
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                            collate_fn=lambda b: collate_codegen(b, tokenizer, config.max_seq_len))

    optimizer = AdamW(model.parameters(), lr=config.lr)
    total_steps = len(train_loader) * config.num_epochs_gen
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)

    model.train()
    global_step = 0
    for epoch in range(config.num_epochs_gen):
        print(f"Epoch {epoch+1}/{config.num_epochs_gen}")
        pbar = tqdm(train_loader)
        for batch in pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            pbar.set_description(f"loss={loss.item():.4f}")
        # run a quick generation evaluation after each epoch
        model.eval()
        gen_results = evaluate_generation(model, tokenizer, val_examples, config)
        model.train()
        print(f"After epoch {epoch+1} - BLEU: {gen_results['bleu']:.4f}  EM: {gen_results['exact_match']*100:.2f}%")
        # save checkpoint
        save_path = os.path.join(config.output_dir, f"{out_prefix}_epoch{epoch+1}")
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)

    # final evaluation
    model.eval()
    final_results = evaluate_generation(model, tokenizer, val_examples, config)
    print("Final generation results:", final_results)
    return model, tokenizer, final_results

In [8]:
def evaluate_generation(model, tokenizer, val_examples, config: Config, max_eval=200):
    """
    Generates code for each prompt in val_examples and computes BLEU and exact-match.
    This is a small quick eval for demonstration; for final work use CodeBLEU.
    """
    hyps = []
    refs = []
    model = model.to(DEVICE)
    with torch.no_grad():
        for i, ex in enumerate(val_examples[:max_eval]):
            prompt = ex["prompt"]
            ref = normalize_code_text(ex["target"])
            input_ids = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
            # generate
            outputs = model.generate(
                input_ids,
                max_new_tokens=config.gen_max_new_tokens,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                do_sample=False,
            )
            # outputs include prompt + generated. decode only the generated tail
            gen_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
            gen_text = normalize_code_text(gen_text)
            hyps.append([tkn for tkn in nltk.word_tokenize(gen_text)])
            refs.append([[tkn for tkn in nltk.word_tokenize(ref)]])
    # BLEU
    chencherry = SmoothingFunction()
    try:
        bleu = corpus_bleu(refs, hyps, smoothing_function=chencherry.method1)
    except Exception:
        # fallback: compute sentence-wise average
        bleu_vals = [nltk.translate.bleu_score.sentence_bleu(r, h, smoothing_function=chencherry.method1) for h, r in zip(hyps, refs)]
        bleu = sum(bleu_vals) / len(bleu_vals)
    # exact match (string equality)
    em_cnt = 0
    for h, r in zip(hyps, refs):
        if " ".join(h).strip() == " ".join(r[0]).strip():
            em_cnt += 1
    exact_match = em_cnt / len(hyps) if hyps else 0.0
    return {"bleu": float(bleu), "exact_match": float(exact_match), "n": len(hyps)}

In [9]:
def make_synthetic_bug_dataset(use_examples, n_train=2000, n_val=400):
    """
    Build a simple synthetic classification dataset:
    - positive (label 0): the original function code
    - negative (label 1): slightly corrupted code introducing an obvious bug (e.g., change '+ to -', remove return)
    This is a toy example for demonstrating fine-tuning & evaluation.
    """
    print("Preparing synthetic bug-detection dataset (toy)...")
    clean_codes = [ex["target"] for ex in use_examples if ex.get("target")]
    # deduplicate
    clean_codes = list(dict.fromkeys(clean_codes))
    # Build pairs
    train = []
    val = []
    def make_buggy(code: str):
        if "return" in code:
            # remove the return statement (simple)
            return code.replace("return", "#BUG_REMOVED return", 1)
        else:
            # flip a + to - if present
            if "+" in code:
                return code.replace("+", "-", 1)
            else:
                return code + "\n# BUG: none detected\n"
    random.shuffle(clean_codes)
    for i, c in enumerate(clean_codes):
        if i >= n_train + n_val:
            break
        buggy = make_buggy(c)
        item_clean = {"code": c, "label": 0}
        item_bug = {"code": buggy, "label": 1}
        if i < n_train:
            train.append(item_clean); train.append(item_bug)
        else:
            val.append(item_clean); val.append(item_bug)
    print(f"Built synthetic dataset: train={len(train)}, val={len(val)}")
    return train, val